# Almacenamiento en cache

El almacenamiento en cache permite que Spark conserve los datos de los cálculos y operaciones.

Almacena el RDD tanto como sea posible en la memoria.

Podemos marcar un RDD almacenado en cache usando **persist()** o **cache()**.

**cache()** es sinónimo de **persist(MEMORY_ONLY)**.

**persist()** puede usar memoria, disco o ambos.

Los niveles de almacenamiento son:

**MEMORY_ONLY**: si cabe en memoria esta es la más rápida al momento de ejecución.

**MEMORY_AND_DISK**

**DISK_ONLY**: no debe usarse al menos que los calculos sean costosos.

**MEMORY_ONLY_2, MEMORY_AND_DISK_2**: tolera fallos

Replica cada partición en dos nodos cluster.


In [ ]:
!pip install findspark

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [ ]:
rdd = sc.parallelize([item for item in range(10)])

In [ ]:
from pyspark.storagelevel import StorageLevel

In [ ]:
rdd.persist(StorageLevel.MEMORY_ONLY)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [ ]:
rdd.unpersist()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [ ]:
rdd.persist(StorageLevel.DISK_ONLY) # Cambiamos el nivel de persistencia

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [ ]:
rdd.unpersist()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [ ]:
rdd.cache()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

# Particionado

Los RDD operan con datos no como una sola masa de datos, sino que administran y operan los datos en particiones repartidas por todo el cluster.

Si la cantidad de particiones es pequeña usaremos solo unos pocos CPU/Nucleos en una gran cantidad de datos, por lo que el rendimiento será lento.

Si la cantidad de particiones es gran se usarán más de los que realmente se necesitan, y en un entorno de múltiples procesos podría provocar falta de recursos para otros miembros

## Particionadores

El particionamiento de los RDD se realiza mediante particionadores, estos asignan un índice a los elementos del RD. Todos los elementos de una partición tendrán el mismo índice de partición.

Spark usa dos:
- **HashPartitioner:** predeterminado, calcula un valor hash para cada clave.
- **RangePartitioner:** divide el RDD en partes aproximadamente iguales.

### Particionador HASH

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [ ]:
rdd = sc.parallelize(['x', 'y', 'z'])

In [ ]:
# INDICE = HASH(ITEM) / NUM_PARTICIONES

In [ ]:
num_particiones = 6

In [ ]:
hash('x') % num_particiones

1

In [ ]:
hash('y') % num_particiones

4

In [ ]:
hash('z') % num_particiones

1

# Shuffling

El movimiento de datos necesarios para el reparticionamiento se denomina **shuffling**.

Determina donde empieza y termina cada unidad particionada.

Se dividen con **groupByKey()**, se filtran y luego de un map queda un RDD final, todo con una relación 1 a 1.

Luego de este stage se genera otro (1) donde se realiza un Shuffle y luego un map.

# Variables broadcast

Las **variables broadcast** son variables compartidas entre todos los ejecutores. Estas se crean en el controlador y luego se leen sólo en los ejecutores.

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [ ]:
rdd = sc.parallelize([item for item in range(10)])

In [ ]:
uno = 1

In [ ]:
broadcast_uno = sc.broadcast(uno)

In [ ]:
rdd1 = rdd.map(lambda x: x + broadcast_uno.value)

In [ ]:
rdd1.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [ ]:
# Un broadcast se destruye con .destroy()

In [ ]:
broadcast_uno.destroy()

In [ ]:
# broadcast_uno.collect()

# Acumuladores

Los **acumuladores** son variables compartidas entre ejecutres que normalmente se utilizan para gregar contadores a su programa Spark.

Con `sparkContext.accumulator()` se pueden definir variables de acumulador.

La función `add()` se usa para agregar/actualizar un valor en el acumulador.

La propiedad `value` de la variable del acumulador se utiliza para recuperar el valor del acumulador.

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

#### Acumuladores con listas de números

In [ ]:
acumulador = sc.accumulator(0) # Empieza en 0

In [ ]:
rdd = sc.parallelize([2, 4, 6, 8, 10])

In [ ]:
rdd.foreach(lambda x: acumulador.add(x))

In [ ]:
print(acumulador.value)

30


#### Acumuladores con listas de String

In [ ]:
rdd1 = sc.parallelize("Cadena para acumulador de spark".split(" "))

In [ ]:
acumulador1 = sc.accumulator(0)

In [ ]:
rdd1.foreach(lambda x: acumulador1.add(1))

In [ ]:
acumulador1.value

5